In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv('/content/IMDB Dataset.csv',sep=',',on_bad_lines='skip')

In [ ]:
data

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [ ]:
data.shape

(50000, 2)

In [ ]:
data.columns

Index(['review', 'sentiment'], dtype='object')

In [ ]:
data.loc[0,'review']

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [ ]:
data.loc[:,'sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [ ]:
# convert positive-->1 , negative-->0
data.loc[:,'Sentiment']=data.loc[:,'sentiment'].map({'positive':1,'negative':0})

In [ ]:
data.drop(columns=['sentiment'],inplace=True)

# Basic Data Cleaning

In [ ]:
import re
def clean_text(text):
  text = text.lower()
  text = re.sub(r'<.*?>','',text)
  text =re.sub(r'[^a-zA-Z!?]',' ',text)
  text = re.sub(r'\s+',' ',text).strip()
  return text

In [ ]:
data.loc[:,'cleaned_text']=data.loc[:,'review'].apply(clean_text)

In [ ]:
data.loc[20000,'cleaned_text']

'i am a huge fan of northern exposure men in trees is a complete knock off of northern exposure there s the city folk from new york stuck in a remote backwoods town marin frist joel fleishman she immediately doesn t hit it off with a local jack slattery maggie o connell the town only has one pilot who is not often available ther is only one radio show and the entire show is about quirky people with sincere and odd relationships too many parallels just like northern exposure except one obvious demographic has been altered in each character i m bored give me something new to think about on the upside the writing is pretty good but if this was a school project it would certainly have earned the stamp of plagiarism perhaps people who love this show were sheltered from the dedicated creativity and hard labors of joshua brand and john falsey'

# Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(data['cleaned_text'])
X = tokenizer.texts_to_sequences(data['cleaned_text'])

In [ ]:
X[0]

[28,
 4,
 1,
 77,
 2037,
 46,
 1051,
 11,
 100,
 149,
 41,
 3056,
 394,
 20,
 229,
 29,
 3171,
 32,
 25,
 204,
 14,
 10,
 6,
 613,
 47,
 590,
 17,
 68,
 1,
 88,
 148,
 11,
 3216,
 68,
 44,
 3056,
 13,
 91,
 2,
 135,
 4,
 559,
 61,
 265,
 8,
 204,
 37,
 1,
 647,
 141,
 1721,
 68,
 10,
 6,
 23,
 3,
 116,
 16,
 1,
 2310,
 40,
 10,
 116,
 2569,
 56,
 17,
 5,
 1455,
 371,
 40,
 559,
 91,
 6,
 3781,
 8,
 1,
 355,
 356,
 4,
 1,
 647,
 7,
 6,
 432,
 3056,
 14,
 11,
 6,
 1,
 357,
 5,
 1,
 2541,
 1031,
 7,
 2683,
 1399,
 22,
 518,
 34,
 4637,
 2435,
 4,
 1,
 1183,
 115,
 30,
 1,
 27,
 2883,
 2,
 385,
 36,
 6,
 23,
 297,
 22,
 1,
 4848,
 2884,
 518,
 6,
 341,
 5,
 107,
 4990,
 2424,
 2,
 52,
 36,
 324,
 2,
 25,
 111,
 223,
 240,
 9,
 60,
 132,
 1,
 280,
 1315,
 4,
 1,
 116,
 6,
 677,
 5,
 1,
 192,
 11,
 7,
 266,
 115,
 77,
 273,
 569,
 21,
 2992,
 819,
 182,
 1292,
 4122,
 16,
 2470,
 1214,
 819,
 1420,
 819,
 865,
 3056,
 152,
 21,
 939,
 184,
 1,
 88,
 394,
 9,
 123,
 210,
 3216,
 68,
 14,
 36,

In [ ]:
y = data['Sentiment']

In [ ]:
# split data for training and testing
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
# padding is used to make length of sentences same
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_train = pad_sequences(x_train,maxlen=100,padding='post',truncating='post')
x_test = pad_sequences(x_test,maxlen=100,padding='post',truncating='post')

# RNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(SimpleRNN(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,417 (2.49 MB)

 Trainable params: 652,417 (2.49 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.fit(x_train,y_train,epochs=10,batch_size=64,verbose=1)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 31s 46ms/step - accuracy: 0.5339 - loss: 0.6896
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 40ms/step - accuracy: 0.6497 - loss: 0.6249
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 41ms/step - accuracy: 0.7099 - loss: 0.5625
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.7800 - loss: 0.4613
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8267 - loss: 0.3777
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8426 - loss: 0.3402
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.8943 - loss: 0.2337
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - accuracy: 0.9254 - loss: 0.1651
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 41ms/step - accuracy: 0.9378 - loss: 0.1273
Epoch 10/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.9181 - loss: 0.1699


In [ ]:
loss,accuracy = model.evaluate(x_test,y_test)
print("Loss: ",loss)
print("Accuracy: ",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5488 - loss: 1.6551
Loss:  1.6551138162612915
Accuracy:  0.548799991607666


# LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(LSTM(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.fit(x_train,y_train,epochs=10,batch_size=64,verbose=1)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 130ms/step - accuracy: 0.7604 - loss: 0.4951
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 74s 119ms/step - accuracy: 0.8396 - loss: 0.3766
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 73s 117ms/step - accuracy: 0.8472 - loss: 0.3594
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 74s 118ms/step - accuracy: 0.8757 - loss: 0.3053
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 119ms/step - accuracy: 0.8954 - loss: 0.2594
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 72s 116ms/step - accuracy: 0.9086 - loss: 0.2303
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 73s 117ms/step - accuracy: 0.9228 - loss: 0.1982
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 118ms/step - accuracy: 0.9359 - loss: 0.1679
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 118ms/step - accuracy: 0.9464 - loss: 0.1455
Epoch 10/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 72s 116ms/step - accuracy: 0.9561 - loss: 0.1226


In [ ]:
loss,accuracy = model.evaluate(x_test,y_test)
print("Loss: ",loss)
print("Accuracy: ",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.8266 - loss: 0.5753
Loss:  0.5753150582313538
Accuracy:  0.8266000151634216


# GRU

In [ ]:
from tensorflow.keras.layers import GRU

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(GRU(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.fit(x_train,y_train,epochs=10,batch_size=64,verbose=1)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 128ms/step - accuracy: 0.7099 - loss: 0.5378
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 75s 121ms/step - accuracy: 0.8548 - loss: 0.3396
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 123ms/step - accuracy: 0.8845 - loss: 0.2826
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 124ms/step - accuracy: 0.9026 - loss: 0.2409
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 81s 123ms/step - accuracy: 0.9234 - loss: 0.1993
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 126ms/step - accuracy: 0.9393 - loss: 0.1631
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 123ms/step - accuracy: 0.9521 - loss: 0.1332
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 77s 122ms/step - accuracy: 0.9630 - loss: 0.1074
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 75s 120ms/step - accuracy: 0.9711 - loss: 0.0878
Epoch 10/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 77s 123ms/step - accuracy: 0.9778 - loss: 0.0690


In [ ]:
loss,accuracy = model.evaluate(x_test,y_test)
print("Loss: ",loss)
print("Accuracy: ",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8251 - loss: 0.7229
Loss:  0.7228702902793884
Accuracy:  0.8251000046730042
